# 🔍 기능 3. Research Gap Analyzer - 공백 매트릭스 합성, 학술 번역 및 비동기 배치 (심화)

본 가이드는 **대규모 문헌 비교 분석기(Research Gap Analyzer)**의 심화 기능인 **연구 공백 매트릭스 합성 및 3대 연구 로드맵 제안, 전문 학술 번역 엔진 구현, Background Tasks 비동기 배치 연산 흐름 및 SSE 푸시 연동**의 핵심 동작 원리를 심도 있게 학습합니다.

실제 다단계 백그라운드 태스크의 핵심 코드와 한국어 번역 가이드라인 프롬프트를 노트북에 맞게 이식하여 실습하도록 설계되었습니다.

---

## 📌 심화 학습 핵심 포인트
1. **연구 공백 매트릭스 합성 (Synthesis)**
   - 개별 논문 분석 결과를 집대성하여 공통적 연구 공백(`common_limitations`)과 이를 해결하기 위한 3가지 혁신적인 로드맵(`suggested_directions`)을 생성합니다.
2. **전문 학술 번역 프롬프트 엔진**
   - Transformer(트랜스포머), RAG 등 고유 명칭의 무조건 직역을 방지하고, 명사구 중심 격식체 종결 방식 및 영문 인용구(`source_quote`) 번역 배제 규칙을 엄격히 적용하는 번역 체인을 구현합니다.
3. **비동기 배치 시뮬레이션**
   - 수십 초가 소요되는 LLM 연산 특성 상 타임아웃을 막기 위한 비동기 배치 상태 업데이트(PENDING -> RUNNING -> COMPLETED) 흐름을 모사합니다.
4. **SSE (Server-Sent Events) 알림 전송 및 리로드 행 해결**
   - 실시간 진행률 공유를 위한 SSE 이벤트 방출 방식 및 uvicorn 셧다운 행 방지 시그널 훅 구조를 분석합니다.

## 1단계. 환경 및 백엔드 설정 로드

환경을 셋업하고 API Key를 바인딩합니다.

In [1]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 현재 notebooks/research_gap/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

load_dotenv(dotenv_path=backend_dir / ".env")
openai_key = os.getenv("OPENAI_API_KEY", "")
print(f"🔑 OpenAI API Key 설정 완료: {'Yes' if openai_key else 'No'}")

🔑 OpenAI API Key 설정 완료: Yes


## 2단계. 연구 공백 매트릭스 합성 및 로드맵 제안

4개 논문의 개별 분석 목록을 기반으로 전체 문헌군을 관통하는 공통 한계점(`common_limitations`)을 도출하고,
이를 극복할 3가지 혁신적 연구 주제(`suggested_directions`)를 정의된 DTO(`ResearchGapMatrix`)로 구조화 합성합니다.

In [2]:
# --- Pydantic 스키마 정의 (DTO) ---
class ExtractionItem(BaseModel):
    summary: str = Field(description="1-sentence summary of the extracted point")
    source_quote: str = Field(description="Verbatim sentence from the text")

class PaperReportItem(BaseModel):
    arxiv_id: str
    title: str
    problems_solved: list[ExtractionItem]
    limitations: list[ExtractionItem]
    similarity: float = 0.0

class SuggestedDirection(BaseModel):
    topic: str = Field(description="Innovative research topic title")
    rationale: str = Field(description="Why this topic is needed to address the gaps")
    proposed_methodology: str = Field(description="Concrete outline of the proposed approach/experiment")

class ResearchGapMatrix(BaseModel):
    papers: list[PaperReportItem] = Field(description="List of analyzed papers")
    common_limitations: list[str] = Field(description="At least 2 identified underlying gaps or common limitations across the papers")
    suggested_directions: list[SuggestedDirection] = Field(description="Exactly 3 proposed future research topics addressing the gaps")

# --- 합성 연산 체인 ---
async def synthesize_gap_matrix(papers_analysis: list[dict], query: str) -> ResearchGapMatrix:
    if not openai_key:
        return ResearchGapMatrix(
            papers=[],
            common_limitations=["Dependency on specialized GPU kernels.", "Limited evaluation in downstream tasks."],
            suggested_directions=[
                SuggestedDirection(
                    topic="Hardware-Agnostic Attention Kernels",
                    rationale="Address CUDA dependencies.",
                    proposed_methodology="Compile with Triton to target multiple platforms."
                )
            ]
            
        )
        
    llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_key, temperature=0.1)
    structured_llm = llm.with_structured_output(ResearchGapMatrix)
    
    # 인풋 가공
    matrix_input_data = "\n\n".join([
        f"Paper: {p['title']} (ID: {p['arxiv_id']})\n"
        f"- Solved: {', '.join([item['summary'] for item in p['problems_solved']])}\n"
        f"- Limitations: {', '.join([item['summary'] for item in p['limitations']])}"
        for p in papers_analysis
    ])
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are a visionary research scientist overseeing the research gap matrix of a specific domain.\n"
            "Review the analysis results of multiple papers, identify the common limitations (underlying gaps), "
            "and propose 3 highly innovative and concrete future research topics (suggested directions) to address those gaps.\n"
            "Respond in English. Structure the response strictly according to the format."
        )),
        ("user", "Research Gap Matrix:\n{matrix_data}\n\nTarget Query/Topic: {query}")
    ])
    
    chain = prompt | structured_llm
    matrix = await chain.ainvoke({
        "matrix_data": matrix_input_data,
        "query": query
    })
    
    if not isinstance(matrix, ResearchGapMatrix):
        raise TypeError(f"Expected ResearchGapMatrix, got {type(matrix)}")
        
    return matrix

# 가상 분석 결과 주입 및 테스트
dummy_analyses = [
    {
        "arxiv_id": "2301.0001",
        "title": "FlashAttention: Fast Attention with IO-Awareness",
        "problems_solved": [{"summary": "Optimizes attention computation speed", "source_quote": "..."}],
        "limitations": [{"summary": "Heavily coupled with NVIDIA CUDA", "source_quote": "..."}]
    },
    {
        "arxiv_id": "2402.0002",
        "title": "Mamba: Linear-Time Sequence Modeling",
        "problems_solved": [{"summary": "Solves quadratic attention complexity", "source_quote": "..."}],
        "limitations": [{"summary": "Performs poorly on strict retrieval tasks", "source_quote": "..."}]
    }
]

synthesized_result = await synthesize_gap_matrix(dummy_analyses, "efficient transformer architectures")
print("🎯 공통 연구 공백 (Gaps):")
for gap in synthesized_result.common_limitations:
    print(f"  - {gap}")
print("\n💡 AI 추천 3대 로드맵 주제:")
for idx, dir_item in enumerate(synthesized_result.suggested_directions, 1):
    print(f"  {idx}. {dir_item.topic}\n     - 배경: {dir_item.rationale}\n     - 방법: {dir_item.proposed_methodology}")

🎯 공통 연구 공백 (Gaps):
  - Limited portability due to reliance on specific hardware (e.g., NVIDIA CUDA)
  - Inadequate performance on strict retrieval tasks, indicating a need for improved accuracy in certain applications.

💡 AI 추천 3대 로드맵 주제:
  1. Cross-Platform Efficient Attention Mechanisms
     - 배경: To overcome the limitations of hardware dependency, there is a need for attention mechanisms that can operate efficiently across various platforms and hardware configurations.
     - 방법: Develop a new attention mechanism that abstracts away from CUDA dependencies, utilizing open-source libraries and optimizing for both CPU and GPU architectures.
  2. Hybrid Retrieval-Optimized Transformer Models
     - 배경: Addressing the performance gap in strict retrieval tasks can enhance the applicability of transformer models in real-world scenarios where accuracy is critical.
     - 방법: Integrate retrieval-augmented generation techniques with existing transformer architectures, employing a dual-model a

/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ResearchGapMatrix(papers=...ptimize performance.")]), input_type=ResearchGapMatrix])
  return self.__pydantic_serializer__.to_python(


## 3단계. 전문 학술 번역 프롬프트 엔진 실습

완성된 영문 연구 공백 리포트를 한국어로 정교하게 번역하되, 아래의 학술 가이드라인 규칙을 충족하는 번역기 코드를 작성합니다.
1. **약어/모델명 유지**: RAG, LLM, MLP 및 Transformer(트랜스포머) 같은 핵심은 직역하지 않고 발음대로 적거나 약어 영문 그대로 둡니다.
2. **종결어미 톤앤매너**: 서술어 끝을 명사구나 nominalized 형식(`~을 해결함`, `~ 최적화`)으로 종결합니다.
3. **영문 팩트 보존**: 각 논문의 원문 인용 필드(`source_quote`)는 번역하지 않고 원래 영어 텍스트 그대로 보존하여 RAG 증거 신뢰성을 훼손하지 않습니다.

In [3]:
async def translate_matrix_to_korean(matrix: ResearchGapMatrix) -> ResearchGapMatrix:
    if not openai_key:
        return matrix # Fallback
        
    llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_key, temperature=0)
    structured_llm = llm.with_structured_output(ResearchGapMatrix)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an expert academic translator specializing in computer science.\n"
            "Translate the given Research Gap Matrix (including paper titles, problems solved, limitations, common limitations, and suggested directions) into natural, professional, and clear Korean.\n\n"
            "Strictly follow these translation guidelines:\n"
            "1. DO NOT translate specific technical model/component names literally if they are widely accepted in English (e.g., 'Transformer' -> '트랜스포머', not '변환기').\n"
            "2. Keep common domain-specific technical terms in English or use standard academic Korean (e.g., keep MLP, RAG, LLM; CMB, redshift, dark matter or write them with Korean translation alongside English abbreviations).\n"
            "3. Use a formal, concise academic tone appropriate for research reports (e.g., ending with nominalized phrases like '~을 해결함', '~을 규명함', '~을 제안함', or clear noun phrases like '~ 최적화', '~ 설계'). Avoid passive verbs.\n"
            "4. Ensure paper titles are translated in a way that sounds natural to Korean researchers while preserving key technical terms.\n"
            "5. DO NOT translate the 'source_quote' field of each paper. Keep the 'source_quote' in its original English verbatim.\n\n"
            "Response must be structured in the requested format."
        )),
        ("user", "{matrix_json}")
    ])
    
    chain = prompt | structured_llm
    translated = await chain.ainvoke({
        "matrix_json": json.dumps(matrix.model_dump(), ensure_ascii=False)
    })
    
    if not isinstance(translated, ResearchGapMatrix):
        raise TypeError(f"Expected ResearchGapMatrix, got {type(translated)}")
        
    # 인용구(source_quote) 유실 차단을 위한 원본 매핑 복원
    for idx, paper in enumerate(translated.papers):
        if idx < len(matrix.papers):
            orig_p = matrix.papers[idx]
            # problems_solved 인용구 원복
            for p_idx, item in enumerate(paper.problems_solved):
                if p_idx < len(orig_p.problems_solved):
                    item.source_quote = orig_p.problems_solved[p_idx].source_quote
            # limitations 인용구 원복
            for l_idx, item in enumerate(paper.limitations):
                if l_idx < len(orig_p.limitations):
                    item.source_quote = orig_p.limitations[l_idx].source_quote
                    
    return translated

# 번역 실행 테스트
translated_korean = await translate_matrix_to_korean(synthesized_result)
print("🇰🇷 한국어 번역 로드맵 추천 주제:")
for idx, dir_item in enumerate(translated_korean.suggested_directions, 1):
    print(f"  {idx}. {dir_item.topic}\n     - 배경: {dir_item.rationale}\n     - 방법: {dir_item.proposed_methodology}")

🇰🇷 한국어 번역 로드맵 추천 주제:
  1. 크로스 플랫폼 효율적인 주의 메커니즘
     - 배경: 하드웨어 의존성의 한계를 극복하기 위해 다양한 플랫폼과 하드웨어 구성에서 효율적으로 작동할 수 있는 주의 메커니즘이 필요함.
     - 방법: CUDA 의존성을 추상화하는 새로운 주의 메커니즘을 개발하고, 오픈 소스 라이브러리를 활용하여 CPU 및 GPU 아키텍처 모두에 최적화함.
  2. 하이브리드 검색 최적화 트랜스포머 모델
     - 배경: 엄격한 검색 작업에서의 성능 격차를 해결하면 정확성이 중요한 실제 시나리오에서 트랜스포머 모델의 적용 가능성을 향상시킬 수 있음.
     - 방법: 기존 트랜스포머 아키텍처와 검색 증강 생성 기법을 통합하여 생성 기반 및 검색 기반 방법을 결합한 이중 모델 접근 방식을 채택함.
  3. 트랜스포머를 위한 적응형 복잡성 감소 기법
     - 배경: 효율성을 더욱 향상시키기 위해 입력 특성에 따라 복잡성을 동적으로 조정하는 적응형 방법에 대한 연구가 필요함.
     - 방법: 실시간으로 입력 시퀀스를 분석하고 주의 메커니즘의 복잡성을 조정하는 프레임워크를 구현하며, 성능 최적화를 위해 강화 학습을 활용할 수 있음.


/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ResearchGapMatrix(papers=...용할 수 있음.')]), input_type=ResearchGapMatrix])
  return self.__pydantic_serializer__.to_python(


## 4단계. BackgroundTasks 비동기 배치 연산 및 상태 추적

FastAPI의 `BackgroundTasks`를 이용해 오랜 연산(약 20초)이 필요한 배치 연산 진행도(`progress`)를 데이터베이스에 단계적으로 갱신하며 클라이언트 폴링에 대응하는 동작 모델의 구조를 학습합니다.
- `PENDING` (0%) -> `RUNNING` (RAG 쿼리 및 임베딩 10%) -> `RUNNING` (유사도 검색 40%) -> `RUNNING` (개별 논문 분석 60%) -> `RUNNING` (매트릭스 합성 80%) -> `COMPLETED` (결과 DB 저장 100%)

In [4]:
import asyncio

class MockTaskTracker:
    def __init__(self):
        self.status = "PENDING"
        self.progress = 0
        self.result = None

    async def run_batch_analysis_simulation(self):
        # 1. 태스크 개시
        self.status = "RUNNING"
        self.progress = 10
        print(f"[State Update] Status: {self.status} | Progress: {self.progress}%")
        await asyncio.sleep(0.5)
        
        # 2. RAG 검색 완료
        self.progress = 40
        print(f"[State Update] Status: {self.status} | Progress: {self.progress}%")
        await asyncio.sleep(0.5)
        
        # 3. 개별 논문 크리틱 추출 완료
        self.progress = 70
        print(f"[State Update] Status: {self.status} | Progress: {self.progress}%")
        await asyncio.sleep(0.5)
        
        # 4. 최종 합성 보고서 및 완성
        self.status = "COMPLETED"
        self.progress = 100
        self.result = "{...Synthesized Matrix JSON...}"
        print(f"[State Update] Status: {self.status} | Progress: {self.progress}%")
        print("🎉 백그라운드 배치 분석 작업 성공!")

tracker = MockTaskTracker()
await tracker.run_batch_analysis_simulation()

[State Update] Status: RUNNING | Progress: 10%
[State Update] Status: RUNNING | Progress: 40%
[State Update] Status: RUNNING | Progress: 70%
[State Update] Status: COMPLETED | Progress: 100%
🎉 백그라운드 배치 분석 작업 성공!


## 5단계. SSE (Server-Sent Events) 브로드캐스트 및 리로드 행 해결

상태가 업데이트될 때마다 SSE 스트리밍 채널로 실시간 완료 푸시 이벤트를 발행하는 원리를 살펴보고,
백엔드 개발 서버(`DEBUG=True` 핫 리로드) 실행 시 Uvicorn이 재시작 대기(Hang) 교착상태에 걸리지 않도록 `SIGINT` / `SIGTERM` 시그널 훅을 사용해 SSE 소켓 스트림 대기를 조기 종료시키는 설계의 필요성을 확인합니다.

In [5]:
print("""
💡 [개발 모드 Uvicorn 셧다운 행(Hang) 해결 시그널 핸들러 훅 스펙]

# backend/main.py lifespan 내 시그널 캡처 코드 예시:

import signal
import asyncio
from api.v1.notification.services import notification_broadcaster

@asynccontextmanager
async def lifespan(app: FastAPI):
    loop = asyncio.get_running_loop()

    def signal_shutdown_handler(*args):
        # Uvicorn 핫 리로드 시그널 발생 즉시 SSE 알림 브로드캐스터 종료
        # 연결된 SSE 스트림 큐를 닫아 대기 제너레이터를 탈출시켜 소켓 행을 해결함
        asyncio.run_coroutine_threadsafe(notification_broadcaster.close(), loop)
        
    # OS 시그널 바인딩
    for sig in (signal.SIGINT, signal.SIGTERM):
        signal.signal(sig, signal_shutdown_handler)

    yield
    
    # 정상 lifespan 종료 단계
    await notification_broadcaster.close()
""")


💡 [개발 모드 Uvicorn 셧다운 행(Hang) 해결 시그널 핸들러 훅 스펙]

# backend/main.py lifespan 내 시그널 캡처 코드 예시:

import signal
import asyncio
from api.v1.notification.services import notification_broadcaster

@asynccontextmanager
async def lifespan(app: FastAPI):
    loop = asyncio.get_running_loop()

    def signal_shutdown_handler(*args):
        # Uvicorn 핫 리로드 시그널 발생 즉시 SSE 알림 브로드캐스터 종료
        # 연결된 SSE 스트림 큐를 닫아 대기 제너레이터를 탈출시켜 소켓 행을 해결함
        asyncio.run_coroutine_threadsafe(notification_broadcaster.close(), loop)

    # OS 시그널 바인딩
    for sig in (signal.SIGINT, signal.SIGTERM):
        signal.signal(sig, signal_shutdown_handler)

    yield

    # 정상 lifespan 종료 단계
    await notification_broadcaster.close()

